In [1]:
# 🚀 Import necessary libraries
import jax
import jax.numpy as jnp
import jax.random as random
from jax import jit, vmap, lax
import optax  # Optimizer library
from functools import partial  # Needed for static JIT arguments

# ✅ Define Recursion Constraints
MAX_RECURSION_DEPTH = 50
DIMENSIONAL_CONSTRAINT = 0.8

# ✅ Define Dynamic Pi & Phi Functions With Constraints
@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / (scale_factor + 1)) * DIMENSIONAL_CONSTRAINT

# ✅ GPU-Optimized DPPU Processing Using `lax.scan`
@partial(jit, static_argnames=["depth"])  # ✅ Mark `depth` as a static argument
def dppu_with_dynamic_pi_phi(x, depth: int, scale_factor=1.0):
    depth = min(depth, MAX_RECURSION_DEPTH)  # ✅ Ensure `depth` is a Python integer

    def body_fn(carry, i):
        x = carry  # Current state
        pi_dynamic = dynamic_pi(i, scale_factor)
        phi_dynamic = dynamic_phi(i, scale_factor)

        scaling = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT
        new_x = jnp.sin(x * scaling * pi_dynamic) * jnp.exp(-x / (phi_dynamic + 1))

        # ✅ Soft constraint to prevent divergence
        new_x = jnp.where(jnp.abs(new_x) > 1e3, x, new_x)

        return new_x, None  # ✅ Return updated value and dummy scan output

    x, _ = lax.scan(body_fn, x, jnp.arange(depth, dtype=jnp.int32))
    return x

# ✅ Vectorized Processing for Batch Computation (Optimized for GPU)
batch_size = 100
data_size = 1000
batch_input = jnp.linspace(0, 10, data_size)
batch_input = jnp.tile(batch_input, (batch_size, 1))

batched_dppu_processing = jit(vmap(lambda x: dppu_with_dynamic_pi_phi(x, depth=10, scale_factor=0.5), in_axes=0))
output_batch = batched_dppu_processing(batch_input)

print("Batch Output Shape:", output_batch.shape)

# ✅ Benchmarking Configuration
import time

NUM_TRIALS = 10
INPUT_SIZE = 1000

# ✅ Warm-up (JIT compile)
_ = dppu_with_dynamic_pi_phi(jnp.ones((INPUT_SIZE,)), depth=10)  # ✅ `depth` is now static

# ✅ Run multiple trials & measure execution time
times = []
for _ in range(NUM_TRIALS):
    start = time.time()
    result = dppu_with_dynamic_pi_phi(jnp.ones((INPUT_SIZE,)), depth=10)
    _ = jax.device_get(result)  # ✅ Ensures computation completes on GPU
    end = time.time()
    times.append(end - start)

# ✅ Compute & Display Benchmark Results
avg_time = sum(times) / len(times)
min_time = min(times)
max_time = max(times)

print(f"\n🔥 GPU-Optimized Benchmark Results (Input Size: {INPUT_SIZE}, Trials: {NUM_TRIALS}) 🔥")
print(f"✅ Average Execution Time: {avg_time:.6f} seconds")
print(f"✅ Fastest Execution Time: {min_time:.6f} seconds")
print(f"✅ Slowest Execution Time: {max_time:.6f} seconds")

# ✅ Display GPU Details
jax.devices()



Batch Output Shape: (100, 1000)

🔥 GPU-Optimized Benchmark Results (Input Size: 1000, Trials: 10) 🔥
✅ Average Execution Time: 0.019259 seconds
✅ Fastest Execution Time: 0.000849 seconds
✅ Slowest Execution Time: 0.170148 seconds


[CudaDevice(id=0)]